# NSE 6-Pattern Unified Scanner (1Y)

**Patterns**: Cup & Handle | Flat Base | Inverse H&S | Double Bottom | Ascending Triangle | Bull Flag

**Features**:
- Pattern start date, pattern end date, expected breakout zone
- Only BUY (moderate+) signals in final output
- Historical CANSLIM + forward returns
- Market cap classification, sector, all metrics

**Runtime**: ~30–45 min for all NSE on Kaggle (1-year lookback is fast)

In [ ]:
!pip install -q yfinance openpyxl scipy

In [ ]:
import os, time, json, pickle
import yfinance as yf
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.signal import find_peaks
from datetime import datetime, timedelta
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

STOCK_LIMIT = None
NIFTY = '^NSEI'
PERIOD = '1y'
MAX_WORKERS = 10
CHECKPOINT_EVERY = 200
SCAN_STEP = 1

TIMEFRAMES = [('Daily', '1d'), ('Weekly', '1wk')]
FWD_HORIZONS = [3, 5, 10, 20, 60]

CANSLIM_CFG = {
    'C_min': 0.25, 'A_min': 0.25, 'N_max_from_high': 0.15,
    'L_min_rs': 1.10, 'I_min_instl': 0.20,
    'buy_strong': 6, 'buy_moderate': 4,
}

print('Config loaded')

## Universe + Nifty + Fundamentals

In [ ]:
def load_universe(limit=STOCK_LIMIT):
    url = 'https://archives.nseindia.com/content/equities/EQUITY_L.csv'
    df = pd.read_csv(url).dropna(subset=['SYMBOL'])
    for col in [' SERIES', 'SERIES']:
        if col in df.columns:
            df = df[df[col].str.strip() == 'EQ']; break
    syms = [s.strip() + '.NS' for s in df['SYMBOL'].astype(str).tolist()]
    return syms[:limit] if limit else syms

def dl(symbol, interval='1d', period=PERIOD):
    try:
        df = yf.download(symbol, period=period, interval=interval,
                         auto_adjust=True, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.dropna(inplace=True)
        return df if len(df) > 30 else None
    except Exception: return None

def dl_fund(symbol):
    try:
        info = yf.Ticker(symbol).info or {}
        return {k: info.get(k) for k in
                ['marketCap','earningsQuarterlyGrowth','earningsGrowth',
                 'heldPercentInstitutions','fiftyTwoWeekHigh','currentPrice',
                 'regularMarketPrice','sector','industry','longName','shortName']}
    except Exception: return {}

stocks = load_universe()
print(f'{len(stocks)} stocks')

nifty_d = dl(NIFTY, '1d'); nifty_w = dl(NIFTY, '1wk')
assert nifty_d is not None and nifty_w is not None
NIFTY_TF = {'Daily': nifty_d, 'Weekly': nifty_w}
print(f'Nifty loaded')

# Fundamentals cache
FC = os.path.join(OUT_DIR, 'fund_cache.pkl')
FUND = pickle.load(open(FC, 'rb')) if os.path.exists(FC) else {}
miss = [s for s in stocks if s not in FUND]
if miss:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(dl_fund, s): s for s in miss}
        for ft in tqdm(as_completed(futs), total=len(futs), desc='Fundamentals'):
            try: FUND[futs[ft]] = ft.result()
            except: FUND[futs[ft]] = {}
    pickle.dump(FUND, open(FC, 'wb'))
print(f'Fundamentals: {sum(1 for v in FUND.values() if v.get("marketCap"))}/{len(FUND)} with mcap')

## Shared Helpers

In [ ]:
def cap_class(mc):
    if mc is None or pd.isna(mc): return 'Unknown', None
    cr = mc / 1e7
    if cr >= 20000: return 'Large Cap', round(cr)
    if cr >= 5000:  return 'Mid Cap', round(cr)
    if cr >= 500:   return 'Small Cap', round(cr)
    return 'Micro Cap', round(cr)

def canslim(close, vol, idx, fund, tf, nifty_close, nifty_ret):
    p = close[idx]
    out = {}
    # N
    lb = min(252 if tf == 'Daily' else 52, idx)
    hi = np.max(close[max(0,idx-lb):idx+1])
    out['N_from_high_%'] = round((hi-p)/hi*100, 2) if hi > 0 else None
    out['N'] = (hi - p) / hi <= CANSLIM_CFG['N_max_from_high'] if hi > 0 else False
    # S
    if vol is not None and idx >= 20:
        to = np.mean(vol[idx-19:idx+1]) * np.mean(close[idx-19:idx+1]) / 1e7
        out['S_turnover_cr'] = round(to, 2); out['S'] = to >= 1.0
    else:
        out['S_turnover_cr'] = None; out['S'] = False
    # L
    rs_lb = 252 if tf == 'Daily' else 52
    if idx >= rs_lb and idx < len(nifty_ret) and not np.isnan(nifty_ret[idx]):
        sr = close[idx] / close[idx - rs_lb] - 1
        nr = nifty_ret[idx]
        rs = (1 + sr) / (1 + nr) if 1 + nr > 0 else np.nan
        out['L_RS'] = round(rs, 2); out['L'] = rs >= CANSLIM_CFG['L_min_rs']
    else:
        out['L_RS'] = None; out['L'] = False
    # M
    sma = 200 if tf == 'Daily' else 40
    if idx < len(nifty_close) and len(nifty_close) >= sma:
        nifty_sma = np.mean(nifty_close[max(0,idx-sma+1):idx+1])
        out['M'] = bool(nifty_close[idx] > nifty_sma)
    else:
        out['M'] = False
    # C, A, I (current snapshot)
    out['C_qtr'] = fund.get('earningsQuarterlyGrowth')
    out['C'] = out['C_qtr'] is not None and out['C_qtr'] >= CANSLIM_CFG['C_min']
    out['A_ann'] = fund.get('earningsGrowth')
    out['A'] = out['A_ann'] is not None and out['A_ann'] >= CANSLIM_CFG['A_min']
    out['I_pct'] = fund.get('heldPercentInstitutions')
    out['I'] = out['I_pct'] is not None and out['I_pct'] >= CANSLIM_CFG['I_min_instl']
    out['Score'] = sum(bool(out.get(k)) for k in 'NASLMIC')
    return out

def fwd_ret(close, idx, nifty_c):
    out = {}
    bp, bn = close[idx], nifty_c[idx] if idx < len(nifty_c) else np.nan
    for h in FWD_HORIZONS:
        fi = idx + h
        if fi < len(close) and bp > 0:
            sr = close[fi] / bp - 1
            out[f'Ret_{h}d_%'] = round(sr * 100, 2)
            if fi < len(nifty_c) and not np.isnan(bn) and bn > 0:
                nr = nifty_c[fi] / bn - 1
                out[f'Alpha_{h}d_%'] = round((sr - nr) * 100, 2)
            else: out[f'Alpha_{h}d_%'] = None
        else:
            out[f'Ret_{h}d_%'] = None; out[f'Alpha_{h}d_%'] = None
    return out

def recommend(status, score, mkt_up):
    bo = 'Breakout' in status
    if bo and score >= CANSLIM_CFG['buy_strong'] and mkt_up:
        return 'BUY — strong'
    if bo and score >= CANSLIM_CFG['buy_moderate']:
        return 'BUY — moderate'
    if not bo and score >= CANSLIM_CFG['buy_strong']:
        return 'WATCH — await breakout'
    if score >= CANSLIM_CFG['buy_moderate']:
        return 'WATCH — mixed'
    return 'AVOID'

## 6 Pattern Detectors

Each returns a dict with:
- `breakout`: bool
- `quality`: float (higher = better)
- `pattern_start_bar`: int (index within window)
- `pattern_end_bar`: int
- `breakout_zone`: float (price level)
- pattern-specific metrics

In [ ]:
# ======== 1. CUP & HANDLE ========
def det_cup(c, v):
    n = len(c)
    if n < 50: return None
    s = pd.Series(c).rolling(5, min_periods=1).mean().values
    ti = int(np.argmin(s))
    if ti < n*0.20 or ti > n*0.80: return None
    lm, rm = np.max(s[:ti+1]), np.max(s[ti:])
    pk, tr = max(lm, rm), s[ti]
    d = (pk - tr) / pk
    if not (0.08 <= d <= 0.55): return None
    sym = abs(lm - rm) / pk
    if sym > 0.22: return None
    rpi = ti + int(np.argmax(s[ti:]))
    if rpi >= n - 2 or s[rpi] < pk * 0.88: return None
    h = s[rpi:]
    if len(h) < 2: return None
    hd = (np.max(h) - np.min(h)) / np.max(h)
    if hd > 0.20 or np.min(h) < (pk+tr)/2*0.92: return None
    cl, hl = rpi + 1, n - rpi
    r = hl / cl
    if not (0.10 <= r <= 0.45): return None
    cy, cx = s[:rpi+1], np.arange(rpi+1)
    try:
        cf = np.polyfit(cx, cy, 2); fit = np.polyval(cf, cx)
        ssr = np.sum((cy-fit)**2); sst = np.sum((cy-np.mean(cy))**2)
        r2 = 1 - ssr/sst if sst > 0 else 0
        if cf[0] <= 0 or r2 < 0.50: return None
    except: return None
    vs = _vol_surge(v, n)
    bo = c[-1] >= pk * 0.97 and (vs >= 1.2 if vs else False)
    return {'pattern': 'CupHandle', 'breakout': bo, 'quality': round(r2 - sym, 3),
            'pattern_start_bar': 0, 'pattern_end_bar': n-1, 'breakout_zone': round(float(pk), 2),
            'cup_depth_%': round(d*100,2), 'symmetry_%': round(sym*100,2),
            'handle_drop_%': round(hd*100,2), 'roundedness_r2': round(r2,3),
            'handle_cup_ratio': round(r,2), 'vol_surge': vs,
            'peak': round(float(pk),2), 'trough': round(float(tr),2), 'last': round(float(c[-1]),2)}

# ======== 2. FLAT BASE ========
def det_flatbase(c, v):
    n = len(c)
    if n < 35: return None
    best = None
    for bl in range(15, min(75, n)+1):
        base = c[-bl:]
        bh, blo = np.max(base), np.min(base)
        br = (bh - blo) / bh if bh > 0 else 1
        if br > 0.20: break
        bs = n - bl
        tl = min(80, bs)
        if tl < 15: continue
        pre = c[bs-tl:bs]
        tg = (pre[-1] - np.min(pre)) / np.min(pre) if np.min(pre) > 0 else 0
        if tg < 0.10: continue
        best = {'bl': bl, 'bh': bh, 'blo': blo, 'br': br, 'tg': tg, 'bs': bs}
    if best is None: return None
    vs = _vol_surge(v, n)
    bo = c[-1] >= best['bh'] * 0.99 and (vs >= 1.2 if vs else False)
    return {'pattern': 'FlatBase', 'breakout': bo, 'quality': round(best['tg'] - best['br'], 3),
            'pattern_start_bar': best['bs'], 'pattern_end_bar': n-1, 'breakout_zone': round(float(best['bh']),2),
            'base_range_%': round(best['br']*100,2), 'prior_trend_%': round(best['tg']*100,2),
            'base_bars': best['bl'], 'vol_surge': vs,
            'peak': round(float(best['bh']),2), 'trough': round(float(best['blo']),2), 'last': round(float(c[-1]),2)}

# ======== 3. INVERSE HEAD & SHOULDERS ========
def det_ihs(c, v):
    n = len(c)
    if n < 40: return None
    prom = 0.015 * np.mean(c)
    try: troughs, _ = find_peaks(-c, prominence=prom, distance=6)
    except: return None
    if len(troughs) < 3: return None
    hc = troughs[(troughs > n*0.20) & (troughs < n*0.80)]
    if len(hc) == 0: return None
    hi = hc[np.argmin(c[hc])]
    lt = troughs[troughs < hi]; rt = troughs[troughs > hi]
    hl = lt[c[lt] > c[hi]]; hr = rt[c[rt] > c[hi]]
    if len(hl) == 0 or len(hr) == 0: return None
    li, ri = hl[-1], hr[0]
    ls, hd, rs_ = c[li], c[hi], c[ri]
    sa = (ls + rs_) / 2
    asym = abs(ls - rs_) / sa
    if asym > 0.18: return None
    hb = (sa - hd) / sa
    if not (0.03 <= hb <= 0.50): return None
    lp = np.max(c[li:hi+1]); rp = np.max(c[hi:ri+1])
    neckline = (lp + rp) / 2
    if ri >= n - 2: return None
    vs = _vol_surge(v, n)
    bo = c[-1] >= neckline * 0.99 and (vs >= 1.2 if vs else False)
    return {'pattern': 'InverseHS', 'breakout': bo, 'quality': round(hb - asym, 3),
            'pattern_start_bar': int(li), 'pattern_end_bar': int(ri), 'breakout_zone': round(float(neckline),2),
            'head_depth_%': round(hb*100,2), 'shoulder_asym_%': round(asym*100,2),
            'neckline': round(float(neckline),2), 'pattern_width': int(ri - li), 'vol_surge': vs,
            'peak': round(float(neckline),2), 'trough': round(float(hd),2), 'last': round(float(c[-1]),2)}

# ======== 4. DOUBLE BOTTOM ========
def det_dbot(c, v):
    n = len(c)
    if n < 30: return None
    prom = 0.02 * np.mean(c)
    try: troughs, _ = find_peaks(-c, prominence=prom, distance=5)
    except: return None
    if len(troughs) < 2: return None
    best = None
    for i in range(len(troughs)):
        for j in range(i+1, len(troughs)):
            sep = troughs[j] - troughs[i]
            if not (10 <= sep <= 150): continue
            p1, p2 = c[troughs[i]], c[troughs[j]]
            diff = abs(p1-p2) / min(p1,p2)
            if diff > 0.08: continue
            mid = np.max(c[troughs[i]:troughs[j]+1])
            la = (p1+p2)/2
            mr = (mid - la) / la
            if mr < 0.06: continue
            if troughs[j] >= n-2: continue
            sc = mr - diff
            if best is None or sc > best['sc']:
                best = {'sc':sc, 'i':troughs[i], 'j':troughs[j], 'p1':p1, 'p2':p2, 'mid':mid, 'diff':diff, 'mr':mr}
    if best is None: return None
    vs = _vol_surge(v, n)
    bo = c[-1] >= best['mid'] * 0.99 and (vs >= 1.2 if vs else False)
    return {'pattern': 'DoubleBottom', 'breakout': bo, 'quality': round(best['sc'], 3),
            'pattern_start_bar': int(best['i']), 'pattern_end_bar': int(best['j']),
            'breakout_zone': round(float(best['mid']),2),
            'bottom_diff_%': round(best['diff']*100,2), 'middle_rise_%': round(best['mr']*100,2),
            'separation_bars': int(best['j']-best['i']), 'vol_surge': vs,
            'peak': round(float(best['mid']),2), 'trough': round(float(min(best['p1'],best['p2'])),2),
            'last': round(float(c[-1]),2)}

# ======== 5. ASCENDING TRIANGLE ========
def det_asctri(c, v):
    n = len(c)
    if not (15 <= n <= 200): return None
    try:
        pks, _ = find_peaks(c, prominence=0.01*np.mean(c), distance=3)
        trs, _ = find_peaks(-c, prominence=0.01*np.mean(c), distance=3)
    except: return None
    if len(pks) < 2 or len(trs) < 2: return None
    pp = c[pks]; res = np.median(pp)
    sp = (np.max(pp) - np.min(pp)) / res if res > 0 else 1
    if sp > 0.04: return None
    tp = c[trs]
    slope = np.polyfit(trs, tp, 1)[0]
    rise = (tp[-1] - tp[0]) / tp[0] if tp[0] > 0 else 0
    if slope <= 0 or rise < 0.015: return None
    if trs[-1] < n * 0.4: return None
    vs = _vol_surge(v, n)
    bo = c[-1] >= res * 0.99 and (vs >= 1.2 if vs else False)
    return {'pattern': 'AscTriangle', 'breakout': bo, 'quality': round(rise - sp, 3),
            'pattern_start_bar': 0, 'pattern_end_bar': n-1, 'breakout_zone': round(float(res),2),
            'resistance_spread_%': round(sp*100,2), 'support_rise_%': round(rise*100,2),
            'top_touches': len(pks), 'bottom_touches': len(trs), 'vol_surge': vs,
            'peak': round(float(res),2), 'trough': round(float(tp[0]),2), 'last': round(float(c[-1]),2)}

# ======== 6. BULL FLAG ========
def det_flag(c, v):
    n = len(c)
    if n < 10: return None
    best = None
    for pl in range(4, min(25, n-3)+1):
        for fl in range(3, min(20, n-pl)+1):
            tot = pl + fl
            if tot > n: break
            pole = c[n-tot:n-fl]; flag = c[n-fl:]
            if pole[0] <= 0: continue
            pg = (pole[-1] - pole[0]) / pole[0]
            if not (0.08 <= pg <= 1.0): continue
            # pole straightness
            x = np.arange(pl)
            try:
                cf = np.polyfit(x, pole, 1); fit = np.polyval(cf, x)
                ssr = np.sum((pole-fit)**2); sst = np.sum((pole-np.mean(pole))**2)
                r2 = 1 - ssr/sst if sst > 0 else 0
            except: continue
            if cf[0] <= 0 or r2 < 0.55: continue
            up = np.sum(np.diff(pole) > 0) / (pl-1) if pl > 1 else 0
            if up < 0.55: continue
            # flag
            fhi, flo = np.max(flag), np.min(flag)
            fd = (pole[-1] - flo) / pole[-1] if pole[-1] > 0 else 1
            if fd > 0.25: continue
            ph = pole[-1] - pole[0]
            fr = (fhi - flo) / ph if ph > 0 else 1
            if fr > 0.70: continue
            q = pg * r2 * up - fd - fr * 0.5
            if best is None or q > best['q']:
                best = {'q': q, 'pl': pl, 'fl': fl, 'pg': pg, 'r2': r2, 'up': up,
                        'fhi': fhi, 'flo': flo, 'fd': fd, 'fr': fr,
                        'pole_start': c[n-tot], 'pole_top': pole[-1]}
    if best is None: return None
    # vol: compare breakout bar to flag avg
    if v is not None and len(v) == n and best['fl'] > 1:
        fv = np.mean(v[n-best['fl']:-1]) if best['fl'] > 1 else np.mean(v[n-best['fl']:])
        vs = round(v[-1] / fv, 2) if fv > 0 else None
    else: vs = None
    bo = c[-1] >= best['fhi'] * 0.995 and (vs is not None and vs >= 1.2)
    pt = best['fhi'] + (best['pole_top'] - best['pole_start'])
    return {'pattern': 'BullFlag', 'breakout': bo, 'quality': round(best['q'],3),
            'pattern_start_bar': n - best['pl'] - best['fl'], 'pattern_end_bar': n-1,
            'breakout_zone': round(float(best['fhi']),2),
            'pole_gain_%': round(best['pg']*100,2), 'pole_r2': round(best['r2'],3),
            'flag_depth_%': round(best['fd']*100,2), 'pole_bars': best['pl'], 'flag_bars': best['fl'],
            'pole_up_bars_%': round(best['up']*100,1), 'price_target': round(float(pt),2),
            'vol_surge': vs, 'peak': round(float(best['fhi']),2), 'trough': round(float(best['flo']),2),
            'last': round(float(c[-1]),2)}

def _vol_surge(v, n):
    if v is None or len(v) < 20: return None
    a = np.mean(v[-20:])
    return round(v[-1] / a, 2) if a > 0 else None

DETECTORS = [det_cup, det_flatbase, det_ihs, det_dbot, det_asctri, det_flag]
WINDOWS = {
    'det_cup': [60,80,120,180,250], 'det_flatbase': [40,60,80,120,180],
    'det_ihs': [60,80,120,180,250], 'det_dbot': [40,60,100,150,200],
    'det_asctri': [30,50,80,120,180], 'det_flag': [15,20,30,40,50,60],
}

print(f'{len(DETECTORS)} detectors loaded')

## Main Scanner

In [ ]:
def scan_stock(symbol):
    fund = FUND.get(symbol, {})
    cc, ccr = cap_class(fund.get('marketCap'))
    nm = fund.get('longName') or fund.get('shortName')
    rows = []
    for tf, intv in TIMEFRAMES:
        df = dl(symbol, intv)
        if df is None or len(df) < 30: continue
        close = df['Close'].values.astype(float)
        vol = df['Volume'].values.astype(float) if 'Volume' in df.columns else None
        dates = df.index
        nifty_a = NIFTY_TF[tf].reindex(dates, method='ffill')
        nc = nifty_a['Close'].values
        rs_lb = 252 if tf == 'Daily' else 52
        nr = np.full(len(nc), np.nan)
        for i in range(rs_lb, len(nc)):
            if nc[i-rs_lb] > 0: nr[i] = nc[i] / nc[i-rs_lb] - 1

        for end in range(30, len(close), SCAN_STEP):
            for det in DETECTORS:
                wins = WINDOWS[det.__name__]
                best = None
                for w in wins:
                    if end < w: continue
                    sc = close[end-w:end]
                    sv = vol[end-w:end] if vol is not None else None
                    res = det(sc, sv)
                    if res is None: continue
                    if best is None or res['quality'] > best['quality']:
                        best = {**res, '_w': w}
                if best is None: continue
                # Compute CANSLIM at this date
                cs = canslim(close, vol, end-1, fund, tf, nc, nr)
                status = 'Breakout Ready' if best['breakout'] else 'Pattern Formed'
                rec = recommend(status, cs['Score'], cs['M'])
                # FILTER: only keep moderate+ buys and watches
                if 'AVOID' in rec: continue

                fw = fwd_ret(close, end-1, nc)
                # Convert pattern_start_bar / pattern_end_bar to actual dates
                win_start_idx = end - best['_w']
                pstart_idx = win_start_idx + best['pattern_start_bar']
                pend_idx = win_start_idx + best['pattern_end_bar']
                pstart_idx = max(0, min(pstart_idx, len(dates)-1))
                pend_idx = max(0, min(pend_idx, len(dates)-1))

                row = {
                    'Stock': symbol.replace('.NS',''), 'Name': nm,
                    'Sector': fund.get('sector'), 'Market_Cap': cc, 'MCap_Cr': ccr,
                    'Pattern': best['pattern'], 'Timeframe': tf,
                    'Pattern_Start': dates[pstart_idx].date(),
                    'Pattern_End': dates[pend_idx].date(),
                    'Found_Date': dates[end-1].date(),
                    'Breakout_Zone': best['breakout_zone'],
                    'Price_At_Detection': best['last'],
                    'Status': status, 'Quality': best['quality'],
                    'Vol_Surge_x': best.get('vol_surge'),
                }
                # add all pattern-specific metrics
                skip = {'pattern','breakout','quality','pattern_start_bar','pattern_end_bar',
                        'breakout_zone','vol_surge','peak','trough','last'}
                for k2, v2 in best.items():
                    if k2 not in skip and not k2.startswith('_'):
                        row[k2] = v2
                row.update({
                    'CANSLIM': cs['Score'],
                    'C': cs['C'], 'A': cs['A'], 'N': cs['N'], 'S': cs['S'],
                    'L': cs['L'], 'I': cs['I'], 'M': cs['M'],
                    'RS_vs_Nifty': cs['L_RS'],
                    **fw,
                    'Recommendation': rec,
                })
                rows.append(row)
    return rows

## Test on 5 stocks first

In [ ]:
test_syms = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'ADANIENT.NS', 'TATAMOTORS.NS']
for s in test_syms:
    if s not in FUND: FUND[s] = dl_fund(s)

t0 = time.time()
trows = []
for s in test_syms:
    r = scan_stock(s)
    print(f'{s}: {len(r)} signals (BUY/WATCH only)')
    trows.extend(r)
print(f'\nTotal: {len(trows)} in {time.time()-t0:.1f}s')
if trows:
    tdf = pd.DataFrame(trows)
    print(tdf['Pattern'].value_counts())
    display(tdf[['Stock','Pattern','Timeframe','Pattern_Start','Pattern_End',
                 'Found_Date','Breakout_Zone','Status','Quality','CANSLIM','Recommendation']].head(20))

## Full scan with checkpoints

In [ ]:
PF = os.path.join(OUT_DIR, 'progress_6pat.json')
RC = os.path.join(OUT_DIR, 'scan_6patterns_1y.csv')

done = set(json.load(open(PF)).get('done',[])) if os.path.exists(PF) else set()
todo = [s for s in stocks if s not in done]
print(f'Remaining: {len(todo)}')

batch = []; t0 = time.time()
for i, sym in enumerate(tqdm(todo, desc='6-pattern scan')):
    try:
        r = scan_stock(sym)
        if r: batch.extend(r)
        done.add(sym)
    except Exception as e: print(f'Err {sym}: {e}')
    if (i+1) % CHECKPOINT_EVERY == 0 or i == len(todo)-1:
        if batch:
            hdr = not os.path.exists(RC) or os.path.getsize(RC) == 0
            pd.DataFrame(batch).to_csv(RC, mode='a', header=hdr, index=False)
            batch = []
        json.dump({'done': sorted(done)}, open(PF, 'w'))
        el = time.time() - t0
        rt = (i+1)/el if el > 0 else 0
        print(f'  [{i+1}/{len(todo)}] {el/60:.0f}m | ETA {(len(todo)-i-1)/rt/60:.0f}m' if rt else '')
if batch: pd.DataFrame(batch).to_csv(RC, mode='a', header=False, index=False)
print('Done.')

## Results

In [ ]:
df = pd.read_csv(RC)
df = df.drop_duplicates(subset=['Stock','Pattern','Timeframe','Found_Date']).reset_index(drop=True)
print(f'{len(df):,} signals (BUY/WATCH only)\n')
print(df['Pattern'].value_counts())
print(); print(df['Recommendation'].value_counts())
print(); print(df['Market_Cap'].value_counts())

# BUY only
buys = df[df['Recommendation'].str.startswith('BUY')]
print(f'\nBUY signals: {len(buys)}')
display(buys[['Stock','Pattern','Timeframe','Pattern_Start','Pattern_End','Found_Date',
              'Breakout_Zone','Price_At_Detection','Status','Quality','CANSLIM',
              'Recommendation']].head(30))

# Excel
out = os.path.join(OUT_DIR, 'scanner_6patterns_BUY_WATCH.xlsx')
with pd.ExcelWriter(out, engine='openpyxl') as w:
    buys.to_excel(w, sheet_name='BUY_signals', index=False)
    df.to_excel(w, sheet_name='All_signals', index=False)
print(f'\nSaved → {out}')

# ========== SEND TO TELEGRAM ==========
import requests

TG_BOT_TOKEN = os.environ.get('TG_BOT_TOKEN', '')
TG_CHAT_ID = os.environ.get('TG_CHAT_ID', '')

def send_telegram_document(file_path, caption="📊 Scanner Results"):
    if not TG_BOT_TOKEN or not TG_CHAT_ID:
        print('[TELEGRAM] No credentials set.')
        return False
    
    url = f'https://api.telegram.org/bot{TG_BOT_TOKEN}/sendDocument'
    try:
        with open(file_path, 'rb') as f:
            files = {'document': f}
            data = {'chat_id': TG_CHAT_ID, 'caption': caption}
            response = requests.post(url, files=files, data=data)
            if response.ok:
                print(f'✅ [TELEGRAM] Sent: {file_path}')
                return True
            else:
                print(f'❌ [TELEGRAM] Error: {response.text}')
                return False
    except Exception as e:
        print(f'❌ [TELEGRAM] Error: {e}')
        return False

# Send the Excel file
if os.path.exists(out):
    send_telegram_document(out, f"📊 6-Pattern Scan Results ({len(df)} signals, {len(buys)} BUY)")
else:
    print('⚠️ Excel file not found')